# core

> `Project` & friends

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## Table of contents

`_fbf3cf89` [**Module setup**](#module-setup)  
`_1d82f33e` - [Imports](#imports)  
`_aa5fa7c7` - [GitHub API](#github-api)  
`_d476a2e5` - [Verbosity](#verbosity)  
`_1d1c6a8c` - [SolveIT domain](#solveit-domain)  
`_b5d7fcb4` - [Output](#output)  
`_2d4d75ba` -- [`Result`](#result)  
`_50f813c0` -- [`Lines(L)`](#linesl)  
`_efbdf781` -- [`FileList(Lines)`](#filelistlines)  
`_new_rename_id` -- [`_rename_template()`](#rename-template)  
`_0db6a50a` [**Class `Project`**](#class-project)  
`_b93d1a6a` [**`Project.new()`**](#projectnew)  
`_a17b0188` [**`Project.sync()`**](#projectsync)  
`_f9c8ed1e` [**`Project.ship()`**](#projectship)  
`_0ecb7e6e` [**`Project.ls()`**](#projectls)  
`_e908abfe` [**Helpers**](#helpers)  
`_9d5246cf` - [Magic methods](#magic-methods)  
`_9e719823` -- [`__getattr__`](#__getattr__)  
`_fd836d45` -- [`__dir__`](#__dir__)  
`_6c8717a7` -- [`_repr_markdown_`](#_repr_markdown_)  
`_480bb662` - [Detection](#detection)  
`_65aba5ea` -- [Owner, repo](#owner-repo)  
`_846e8d38` -- [Project type](#project-type)  
`_bf0c525a` -- [SolveIT URL](#solveit-url)  
`_1c555637` - [Version](#version)  
`_671c0239` - [GH Pages branch select](#gh-pages-branch-select)  
`_a84a675d` - [Quarto `dark`|`light`](#quarto-darklight)  
`_294a4299` [**TODO**](#todo)

See also: [README](https://1iis.github.io/pj/), [beginner help]().

## Module setup

### Imports

In [ ]:
#| export
#| export
from pathlib import Path
import os, re, subprocess, time
from fastcore.basics import store_attr, patch
from fastcore.foundation import L
from urllib.parse import quote
from fastgit import Git
from ghapi.all import GhApi
from nbdev.release import release_pypi, push_release, bump_version

### GitHub API

Module-level `_default_api`: call `set_api(GhApi())` once; it's shared by all `Project` instances.

In [ ]:
#| export
#| export
_default_api = None  # Module-level API holder

In [ ]:
#| export
#| export
def set_api(api: GhApi|None):
    """Set the default GhApi instance for all Project instances."""
    global _default_api; _default_api = api

In [ ]:
#| export
#| export
def get_api() -> GhApi|None:
    """Get the current default GhApi instance."""
    return _default_api

### Verbosity

Set verbosity once per dialog: `0`=quiet, `1`=normal (default), `2`=verbose.

In [ ]:
#| export
#| export
_verbosity = 1  # 0=quiet, 1=normal, 2=verbose
def set_verbosity(v: int): global _verbosity; _verbosity = v

### SolveIT domain

Cached at import for building file browser URLs.

In [ ]:
#| export
_solveit_domain = os.environ.get('PRIVATE_DOMAIN', '') if os.environ.get('IN_SOLVEIT') else ''

### Output

#### `Result`

Methods return a `Result` instance wrapping the value.
- Value is "**stdout**", composable.
- Display is "**stderr**" (`_repr_markdown_`): human status, notebook-friendly.
- <details><summary>To ease composition, <code>Result.__getattr__</code> forwards to <code>.val</code>.</summary>
  <p><code>Result</code> is invisible: <code>Project.new("foo").sync()</code> chains naturally.</p></details>

In [ ]:
#| export
#| export
class Result:
    def __init__(self, val, ok: bool=True, msg: str=''): store_attr()
    def __repr__(self): return repr(self.val)
    def __getattr__(self, k):
        if k in ('val', 'ok', 'msg') or k.startswith('_'): raise AttributeError(k)
        return getattr(self.val, k)
    def _repr_markdown_(self):
        if _verbosity == 0: return ''
        return f"{'✓' if self.ok else '✗'} {self.msg}"

#### `Lines(L)`

Base class for pretty list output. Inherits from `L` for composition; `_repr_markdown_` joins items as lines.

In [ ]:
#| export
#| export
class Lines(L):
    """L subclass with markdown repr that joins lines."""
    # def _repr_markdown_(self): return '\n'.join(str(x) for x in self) or "Empty."
    def _repr_markdown_(self): return f"```\n{chr(10).join(str(x) for x in self)}\n```" if self else "Empty."


#### `FileList(Lines)`

Returned by `ls()`. Inherits from `L` so it's iterable, indexable, and composable (`for f in pj.ls()`, `.filter()`, etc.), while `_repr_markdown_` provides rich notebook display with SolveIT links for notebooks.

In [ ]:
#| export
#| export
class FileList(Lines):
    def __init__(self, items, base_path, domain=None):
        super().__init__(items)
        self.base_path, self.domain = base_path, domain
    
    def _repr_markdown_(self):
        def _fmt(f):
            rel = f.relative_to(self.base_path)
            if self.domain and f.suffix == '.ipynb':
                rel_home = f.resolve().relative_to(Path.home()).with_suffix('')
                return f"- <a href='https://{self.domain}.solve.it.com/dialog_?name={quote(str(rel_home))}' target='_blank'>`{rel.with_suffix('')}`</a>"
            return f"- `{rel}`"
        return '\n'.join(_fmt(f) for f in self) or "No files found."

### Rename template

`_rename_template()` converts template placeholders (`my_package`, `my-package`) to the actual project name after spawning from a template.

In [ ]:
#| export
def _rename_template(path: Path, name: str, desc: str | None = None) -> None:
    """Rename template placeholders (my_package, my-package) to actual project name."""
    pkg = name.replace('-', '_')
    src = path / 'src' / 'my_package'
    dst = path / 'src' / pkg
    if src.exists(): src.rename(dst)
    for f in [path / 'pyproject.toml', path / 'tests' / 'test_main.py']:
        if not f.exists(): continue
        txt = f.read_text(encoding='utf-8')
        txt = txt.replace('my-package', name).replace('my_package', pkg)
        if desc is not None:
            txt = txt.replace('"One thing, done well."', f'"{desc}"')
        f.write_text(txt)

## Class `Project`

Each `Project` instance represents a repository. Auto-detects owner and repo from the git remote URL. If set, `org` overrides owner.

> [!TIP]
> Existing projects are loaded by instanciating the class with a path (`p = Project('path/to/repo')`).  
> New ones should be created with `p = Project.new()`.  
> 
> Both return a `Project` ready to use.

In [ ]:
#| export
#| export
class Project:
    """Manage a project's lifecycle: init, sync, ship."""
    def __init__(self, path: str|Path, api: GhApi|None=None, org: str|None=None):
        store_attr()  # auto-store params as self.x
        self.path = Path(path)
        self.api = api or _default_api or GhApi()
        self.g = Git(self.path)
        self.owner, self.repo = self._parse_remote()
        if org: self.owner = org

## `Project.new()`

Creates everything from scratch: GitHub repo via API, local clone, nbdev hooks installed. If `nbdev=True`, runs scaffolding and applies dark theme. Initial commit, push, and Pages setup.

In [ ]:
#| export
#| export
@classmethod
def new(cls, 
        name: str,                   # Repository name
        path: str|Path | None=None,  # Local path (defaults to ./name)
        api: GhApi     | None=None,  # GitHub API instance
        org: str       | None=None,  # Organization (overrides authenticated user)
        desc: str      | None=None,  # Repository description
        private: bool=True,          # Create private repository
        nbdev: bool=False,           # Initialize as nbdev project
        template: str|None=None,     # Template name (e.g. "py"), or "owner/repo"
        template_org: str="1iis",    # Template owner (used if template has no "/")
) -> 'Project':
    """Create a new GitHub repo, clone it, and return a Project instance."""
    api = api or _default_api or GhApi()
    owner = org or api.users.get_authenticated().login
    path = Path(path or '.') / name

    if template:
        if nbdev:
            import warnings
            warnings.warn("template overrides nbdev=True; ignoring nbdev")
        if '/' in template:
            tmpl_owner, tmpl_repo = template.split('/', 1)
        else:
            tmpl_owner, tmpl_repo = template_org, template
        api.repos.create_using_template(
            tmpl_owner, tmpl_repo,
            name=name, description=desc or '',
            private=private,
        )
        Git('.').clone(f"git@github.com:{owner}/{name}.git", str(path))
        _rename_template(path, name, desc)
        subprocess.run(['uv', 'sync', '--group', 'dev'], cwd=path)
    else:
        if org: api.repos.create_in_org(org, name=name, description=desc or '', private=private)
        else: api.repos.create_for_authenticated_user(name=name, description=desc or '', private=private)
        Git('.').clone(f"git@github.com:{owner}/{name}.git", str(path))
        subprocess.run(['nbdev_install_hooks'], cwd=path, check=True)
        if nbdev: subprocess.run(['nbdev_new', '--repo', name, '--user', owner], cwd=path, check=True)
    
    proj = cls(path, api=api, org=org)
    if nbdev and not template: proj.dark_theme()
    proj.g.add('-A')
    proj.g.commit(m='Initial commit', mute_errors=True)
    push_ok = True
    try:
        proj.g.push()
    except Exception as e:
        push_ok = False
        if _verbosity >= 1: print("⚠️ Push failed (likely temporary GitHub issue).\n   Repo created successfully. Retry: `p.push()`")
    
    if nbdev and not template and push_ok: proj.setup_pages()
    if os.environ.get('IN_SOLVEIT'): display(proj)
    if _verbosity >= 1: print(f"✓ Project '{name}' created successfully.")
    return proj

Project.new = new

## `Project.sync()`

`nbdev_prepare` first for notebooks, then Git `add` all, `commit`, `pull --rebase`, and `push`.  
We use `rebase` for a clean, linear log (no merge commit) whenever there's no conflict (e.g. one may edit different files on different hosts, and commit all later).

In [ ]:
#| export
#| export
@patch
def sync(self: Project,
         msg: str="sync"  # Git commit message
) -> Result:
    """Commit all, pull --rebase, push. Runs nbdev_prepare if nbdev project."""
    if self._detect_type() == 'nbdev' and os.environ.get('IN_SOLVEIT'): subprocess.run(['nbdev_prepare'], cwd=self.path, check=True)
    self.g.add('-A')
    self.g.commit(m=msg, mute_errors=True)
    self.g.pull(rebase=True)
    self.g.push()
    return Result(msg, msg=f"Synced: {msg}.")

## `Project.ship()`

Full release cycle, auto-detecting strategy by project type. Won't ship with uncommitted changes unless `force=True`.

| Project Type | Version Bump | Build & Upload | Tag & Release |
|--------------|--------------|----------------|---------------|
| **nbdev** | `nbdev_bump_version` | `nbdev_pypi` | git tag + `ghapi` |
| **python** | manual in `pyproject.toml` | `build` + `twine` | git tag + `ghapi` |
| **other** | n/a | n/a | git tag + `ghapi` |


In [ ]:
#| export
#| export
@patch
def ship(self: Project, 
         part: int=2,              # Version part to bump: 0=major, 1=minor, 2=patch
         dry_run: bool=False,      # Show what would happen without executing
         force: bool=False,        # Ship even with uncommitted changes
         pypi: bool=False,         # Upload to PyPI
         quiet: bool=False         # Suppress build output
) -> Result:
    """Bump version, build, upload to PyPI, tag, and create GitHub release."""
    ptype = self._detect_type()
    
    if self.g.status('--porcelain') and not force:
        raise RuntimeError("Uncommitted changes! Commit first or use force=True.")
    
    version = self._get_version()
    if version and ptype != 'other':
        new_version = bump_version(version, part)
        if not dry_run: self._set_version(new_version)
        version = new_version
    
    tag = f"v{version}" if version else self._next_tag()
    if dry_run: return Result(tag, msg=f"Dry run: {tag}.")
    
    if pypi and ptype in ('nbdev', 'python'):
        os.chdir(self.path)
        release_pypi(quiet=quiet)
    
    self.sync(msg=f"Release {tag}")
    if ptype == 'nbdev':
        os.chdir(self.path)
        push_release()
    else:
        self.g.tag(tag, a=True, m=f"Release {tag}")
        self.g.push(tags=True)
        self.api.create_release(owner=self.owner, repo=self.repo, tag_name=tag, generate_release_notes=True)
    
    if ptype == 'nbdev': self._ensure_pages()
    return Result(tag, msg=f"Shipped: {tag}.")

## `Project.ls()`

List repo files with smart defaults. Shortcuts: `'nbs'`, `'py'`; or pass regex. Notebooks get SolveIT links.

In [ ]:
#| export
#| export
@patch
def ls(self: Project, 
       pattern: str = '',      # Shortcut ('nbs', 'py') or regex
       exclude: list = None    # Paths to exclude
) -> FileList:
    """List files in repo, optionally filtered. Returns FileList."""
    shortcuts = {'nbs': r'\.ipynb$', 'py': r'\.py$'}
    regex = shortcuts.get(pattern, pattern) or None
    exclude = exclude or ['.git', '__pycache__', '.ipynb_checkpoints', '_proc', r'^tmpfgm_.*\.ipynb$']
    
    files = sorted(f for f in self.path.rglob('*') if f.is_file())
    for ex in exclude: files = [f for f in files if not re.search(ex, str(f.relative_to(self.path)))]
    if regex: files = [f for f in files if re.search(regex, str(f))]
    
    return FileList(files, self.path, _solveit_domain)


## Helpers

### Magic methods

#### `__getattr__`

Forward unknown method calls to `self.g` (the `Git` instance), so `p.status()` is `p.g.status()` (i.e. `git status`).

In [ ]:
#| export
#| export
#| hide
@patch
def __getattr__(self: Project, k):
    """Forward unknown methods to Git instance, wrapping list outputs in Lines."""
    if k.startswith('_'): raise AttributeError(k)
    attr = getattr(self.g, k)
    if not callable(attr): return attr
    return lambda *a, **kw: Lines(r) if isinstance(r := attr(*a, **kw), list) else r


#### `__dir__`

Extend tab-completion to include `Git` methods. Without this, the editor wouldn't know `p.status` exists.

In [ ]:
#| export
#| export
#| hide
@patch
def __dir__(self: Project): return super().__dir__() + dir(self.g)

#### `_repr_markdown_`

Called by Jupyter/SolveIT when displaying an object: return a markdown string showing project info.

In [ ]:
#| export
#| export
#| hide
@patch
def _repr_markdown_(self: Project): return f"""Project info

Owner: **<a href="https://github.com/{self.owner}" target="_blank">{self.owner}</a>**  
Repo: **<a href="https://github.com/{self.owner}/{self.repo}" target="_blank">`{self.repo}`</a>**  

Type: {self._detect_type()}  
{f'Docs: <a href="https://{self.owner}.github.io/{self.repo}" target="_blank">https://{self.owner}.github.io/{self.repo}</a>' if self._detect_type() == 'nbdev' else ""}  

{f"📁 SolveIT: <a href='{self.solveit_url.rsplit('/', 1)[0]}' target='_blank'><code>[{self.path.parent.resolve()}/]</code></a>**<code><a href='{self.solveit_url}' target='_blank'>{self.path.name}</code></a>**" if _solveit_domain else ""}  
"""

### Detection

#### Owner, repo

`_parse_remote()` extracts `(owner, repo)` from the git remote URL, handling both HTTPS and SSH formats.

In [ ]:
#| export
#| export
#| hide
@patch
def _parse_remote(self: Project):
    """Parse (owner, repo) from git remote URL, or (None, None) if not found."""
    try:
        url = self.g.remote('get-url', 'origin').replace('.git', '')
        parts = url.split('/')
        return parts[-2].split(':')[-1], parts[-1]  # handles git@github.com:owner/repo
    except: return None, None

#### Project type

`_detect_type()` checks which config files exist: `settings.ini` means nbdev, `pyproject.toml` means standard Python, otherwise other.  
Used to make property `Project.pjtype`.

In [ ]:
#| export
#| export
#| hide
@patch
def _detect_type(self: Project):
    """Detect project type: 'nbdev', 'python', or 'other'."""
    if (self.path / 'settings.ini').exists(): return 'nbdev'
    if (self.path / 'pyproject.toml').exists(): return 'python'
    return 'other'

In [ ]:
#| export
#| export
#| hide
@property
def pjtype(self: Project): return self._detect_type()

Project.pjtype = pjtype

#### SolveIT URL

Returns the SolveIT file browser URL for this project, or `None` if not in SolveIT.

In [ ]:
#| export
@property
def solveit_url(self: Project):
    if not _solveit_domain: return None
    rel_path = str(self.path.resolve()).replace('/app/data/', '').replace('/', '%2F')
    return f"https://{_solveit_domain}.solve.it.com/?at={rel_path}"

Project.solveit_url = solveit_url

### Version

In [ ]:
#| export
#| export
#| hide
@patch
def _get_version(self: Project):
    """Extract version from settings.ini (nbdev) or pyproject.toml (standard)."""
    if (p := self.path / 'settings.ini').exists():
        for line in p.read_text().splitlines():
            if line.startswith('version'): return line.split('=')[1].strip()
    if (p := self.path / 'pyproject.toml').exists():
        import tomllib
        return tomllib.loads(p.read_text()).get('project', {}).get('version')
    return None

In [ ]:
#| export
#| export
#| hide
@patch
def _set_version(self: Project, version: str):
    """Write version to settings.ini or pyproject.toml."""
    if (p := self.path / 'settings.ini').exists():
        p.write_text(re.sub(r'^version\s*=.*$', f'version = {version}', p.read_text(), flags=re.MULTILINE))
    elif (p := self.path / 'pyproject.toml').exists():
        p.write_text(re.sub(r'^version\s*=\s*"[^"]*"', f'version = "{version}"', p.read_text(), flags=re.MULTILINE))

In [ ]:
#| export
#| export
#| hide
@patch  
def _next_tag(self: Project) -> str:
    """Generate next tag for non-versioned projects (v1, v2, ...)."""
    tags = self.api.list_tags(owner=self.owner, repo=self.repo)
    nums = [int(t.name[1:]) for t in tags if t.name.startswith('v') and t.name[1:].isdigit()]
    return f"v{max(nums, default=0) + 1}"

### GH Pages branch select

For nbdev:
- `_ensure_pages()` configures GitHub Pages to deploy from the `gh-pages` branch: polls for existence (waiting for Actions), then calls the API.  
- `setup_pages()` for manual call.

In [ ]:
#| export
#| export
#| hide
@patch
def _ensure_pages(self: Project, timeout: int=120, poll_interval: int=10) -> bool:
    """Ensure GitHub Pages is configured for gh-pages branch. Returns True if successful."""
    try:  # Check current config
        pages = self.api.repos.get_pages(owner=self.owner, repo=self.repo)
        if pages.source.branch == 'gh-pages': return True
    except: pass  # Pages not enabled yet
    deadline = time.time() + timeout  # Wait for gh-pages branch to exist
    if _verbosity >= 1: print(f"Waiting for GitHub Actions to create branch: 'gh-pages'. (timeout: {timeout}s)")
    while time.time() < deadline:
        branches = [b.name for b in self.api.repos.list_branches(owner=self.owner, repo=self.repo)]
        if 'gh-pages' in branches: break
        time.sleep(poll_interval)
    else: return False  # Timed out
    
    try:  # Configure Pages
        self.api.repos.create_pages_site(owner=self.owner, repo=self.repo, source={'branch': 'gh-pages', 'path': '/'})
        return True
    except:
        try:  # Maybe already exists, just needs update
            self.api.repos.update_information_about_pages_site(
                owner = self.owner, repo = self.repo, source = {'branch': 'gh-pages', 'path': '/'} )
            return True
        except: return False

In [ ]:
#| export
#| export
@patch
def setup_pages(self: Project) -> Result:
    """Manually configure GitHub Pages. Useful for debugging."""
    if self._ensure_pages(): return Result(True, msg=f"Pages configured: https://{self.owner}.github.io/{self.repo}/")
    return Result(False, ok=False, msg="⚠️  Timeout waiting for gh-pages branch. Run `p.setup_pages()` manually after Actions complete.")


### Quarto `dark`|`light`

Quarto feature: auto-select `light` or `dark` theme based on user system settings, and display a manual toggle button.

1. Modify `nbs/_quarto.yml` to use a `light`|`dark` theme pair.

    ```yaml
        theme:
          light: cosmo
          dark: [cosmo, dark.scss]
    ```

2. Create `nbs/dark.scss` to produce a dark themed nbdev style, with readable font colors for code blocks.

In [ ]:
#| export
#| export
#| echo: false
_dark_scss = '''/*-- scss:defaults --*/
// Quarto adaptation for nbdev theme

// Base document colors for dark mode
$body-bg: #181818;
$body-color: #ccc;
$link-color: #75AADB;

// Code blocks
$code-block-bg-alpha: -.9;

// Navbar
$navbar-bg: #2a2a2a;

/*-- scss:rules --*/

// Fix cell output text visibility in dark mode
.cell-output,
.cell-output-display {
  color: #ccc;
}

.cell-output pre {
  color: #ccc;
  background-color: #1e1e1e;
}

// Ensure code output is visible
.cell-output > pre code,
.cell-output-stdout pre {
  color: #ccc;
}

// Fix inline code in dark mode
code:not(pre > code) {
  background-color: #2a2a2a !important;
  color: #ab8dff;
}

// Fix blockquote code blocks in dark mode
blockquote pre,
blockquote pre code {
  color: #ccc;
  background-color: #1e1e1e;
}

blockquote {
  border-left-color: #555;
  color: #bbb;
}
'''

In [ ]:
#| export
#| export
@patch
def dark_theme(self: Project) -> Result:
    """Apply dark mode theme to nbdev docs."""
    nbs = self.path / 'nbs' # nbdev standard notebook dir
    if not nbs.exists(): raise FileNotFoundError(f"No nbs/ directory in {self.path}")
    quarto_yml = nbs / '_quarto.yml'
    text = re.sub(r'(\s+)theme:\s*cosmo', r'\n    theme:\n      light: cosmo\n      dark: [cosmo, dark.scss]', quarto_yml.read_text())

    quarto_yml.write_text(text)
    (nbs / 'dark.scss').write_text(_dark_scss)
    return Result(True, msg="Dark theme applied.")

## TODO

- We may extend `__getattr__` to forward to the GitHub API (`GhApi`), after `Git` (collisions are rare, hopefully), if/when people want more GH API access.  
Alternatively, we'd build more `pj` methods to handle the most common use-cases (like PRs, reviews, merge, etc.) right from a project's dialog.

- better general outputs
  - better messages for `Project` methods.
  - interactive HTML menu for some of them, with dropdown menus, spoiler folders, info, all the cool stuff.
    - fence that with args, we shouldn't always force such rich stuff.

- richer `ls()` output with links to all the things: source on GH, doc page, SolveIT file editor, Codespaces, VSCode (local), whatever we can do to make it super nice to use.
  - make it optional, with great defaults
  - see user.cfg

- see if we can use L elsewhere now that we're loading it! :D

- think about externalizing the whole display part to its own nb
  - or even its own module, idk. Depends how this generalizes to what, how we want to use it, etc.

- user.cfg
  - some way to have personal settings for all the things, default values, etc.
  - lets you tell pj what you have, what you do, how, etc.
  - e.g. `py` files should open in `...`, whereas X in Y, etc.
  - see if maybe there are nice online tools to use (e.g. nice renderers for file types like ?ML, cool tools to manipulate stuff)
  - this isn't pj specific, so we should have instance-wide user.cfg; then overriding additions of all `user.cfg` files in subdirs (such that each dir inherits from configs between it and root).
  - plug CRAFT and TEMPLATE deploy from central repo/repos (and make that repo, btw!)

- in-dialog editor! (IDE!!! LOL)
  - either an `<iframe>` if we can help it (with the SolveIT editor, or whatever else we can do)
  - or a sol-flow:
    1. run fn to make cell with file content (type raw, note, code; depending on file format)
    2. edit cell (optionally dup it)
    3. run fn to take content from CELL ABOVE and write to file
       - optional "backup" arg to copy first to `old.file` before writing it
       - optional output of result (run first fn to make cell below)
    4. optional: 
       - diff before writing
       - diff after writing (if backup)
       - use Markdown ` ``` ` to fence and render code properly when not python (alt. to raw)
       - clean up: fence with XML raws, confirm, delete. (user-chosen and never surprising)
  